In [0]:
from pyspark.sql import functions as F

# Load the union of raw green taxi tables
tables = ["nyc_mobility.raw.green_03_2026", "nyc_mobility.raw.green_04_2026", "nyc_mobility.raw.green_05_2026"]
dfs = [spark.table(t) for t in tables]
df_raw = dfs[0]
for d in dfs[1:]:
    df_raw = df_raw.unionByName(d)

# add trip duration for use in rules
df_raw = df_raw.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("lpep_dropoff_datetime") - F.unix_timestamp("lpep_pickup_datetime")) / 60
)

In [0]:
# STEP 1 — hard excludes: rows that are not usable trips at all
excluded = df_raw.filter(
    (F.col("lpep_pickup_datetime") < "2026-03-01") |
    (F.col("lpep_pickup_datetime") >= "2026-06-01") |
    (F.col("lpep_dropoff_datetime") < F.col("lpep_pickup_datetime")) |
    (F.col("trip_distance") > 100000)
)

df_kept = df_raw.exceptAll(excluded)

In [0]:
from pyspark.sql.types import IntegerType, DoubleType, TimestampType, StringType, DecimalType

# STEP 2 — cast to correct types and standardize decimals
money_cols = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
              "ehail_fee", "improvement_surcharge", "total_amount",
              "congestion_surcharge", "cbd_congestion_fee"]

int_cols = ["VendorID", "RatecodeID", "PULocationID", "DOLocationID",
            "passenger_count", "payment_type", "trip_type"]

df_typed = df_kept

# integers
for c in int_cols:
    df_typed = df_typed.withColumn(c, F.col(c).cast(IntegerType()))

# timestamps
for c in ["lpep_pickup_datetime", "lpep_dropoff_datetime"]:
    df_typed = df_typed.withColumn(c, F.col(c).cast(TimestampType()))

# distance: double, 2 decimal places
df_typed = df_typed.withColumn(
    "trip_distance", F.round(F.col("trip_distance").cast(DoubleType()), 2)
)

# money fields: double, 2 decimal places
for c in money_cols:
    df_typed = df_typed.withColumn(c, F.round(F.col(c).cast(DoubleType()), 2))

df_typed = df_typed.withColumn("trip_duration_min", F.col("trip_duration_min").cast(DecimalType(10, 2)))

# flag column
df_typed = df_typed.withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag").cast(StringType()))

In [0]:
# STEP 2 — write to silver
target_table_silver = "nyc_mobility.clean.green_taxi"

df_kept.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table_silver)